Generate, evaluate, optimize tweet Using Iterative Workflwo

In [83]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage,HumanMessage
import operator
import os

In [84]:
llm = ChatOpenAI()

In [85]:
class TweetEvaluation(BaseModel):
    evaluation: Literal['approved','needs_improvement'] = Field(description='evaluation of the tweet')
    feedback: str = Field(description='Feedback of the tweet')

llm_with_structured_op = llm.with_structured_output(TweetEvaluation)

In [86]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal['approved','needs_improvement']
    feedback:str
    iteration :int
    max_iteration:int
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [87]:
def generate_tweet(state: TweetState):

    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # send generator_llm
    response = llm.invoke(messages).content


    # return response
    return {'tweet': response, 'tweet_history': [response]}

In [88]:
def evaluate_tweet(state: TweetState):

    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = llm_with_structured_op.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [89]:
def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [90]:
def check_evaluation(state:TweetState) ->str:
    """This function checks the evaluation and returns a evaluation response"""
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration'] : 
        return 'approved'
    else:
        return 'needs_improvement'

In [91]:
graph=StateGraph(TweetState)

graph.add_node('generate_tweet', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START,'generate_tweet' )
graph.add_edge('generate_tweet','evaluate' )
graph.add_conditional_edges('evaluate',check_evaluation, {'approved': END,'needs_improvement' : 'optimize'} )
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [93]:
initial_state = {'topic':'agtuyu','iteration':1, 'max_itertaion':5}
output_state = workflow.invoke(initial_state)
print(output_state)

{'topic': 'agtuyu', 'tweet': "AGTUYU: the sound you make when you see your bank account after a weekend of living your best life. It's the new financial indicator we all need in our lives. AGTUYU levels: dangerously low. #adulting #budgeting #oops", 'evaluation': 'approved', 'feedback': "This tweet is approved. It offers a fresh take on the experience of checking one's bank account after a weekend of spending, introducing the humorous concept of 'AGTUYU levels' as a new financial indicator. The tweet is original, funny, punchy, and relatable to many young adults navigating adulting and budgeting. Its clever use of a made-up acronym adds to its virality potential, making it likely to be shared by those who resonate with the message.", 'iteration': 1, 'tweet_history': ["AGTUYU: the sound you make when you see your bank account after a weekend of living your best life. It's the new financial indicator we all need in our lives. AGTUYU levels: dangerously low. #adulting #budgeting #oops"], 